# KAMA + Markov Regimes — VNINDEX

Implement the core idea from Piotr Pomorski's article [Where Markov meets a moving average](https://piotrpomorski.substack.com/p/where-markov-meets-a-moving-average) on **VNINDEX**, while keeping the data-loading pattern aligned with `notebooks/kronos_vnindex.ipynb`.

This notebook intentionally does **not** reuse the article's code. It rebuilds the workflow with a cleaner structure and a few practical improvements:

| Step | Description |
|------|-------------|
| **1** | Load VNINDEX OHLCV from Delta Lake / HDF5 cache |
| **2** | Fit a 2-state Markov-switching model on log returns to infer volatility regimes |
| **3** | Compute an adaptive trend overlay with Kaufman's Adaptive Moving Average (KAMA) |
| **4** | Combine volatility + trend into 4 market regimes |
| **5** | Visualise regimes on top of price and state probabilities |
| **6** | Backtest the regime rule with transaction costs |
| **7** | Run a walk-forward evaluation using filtered probabilities |

Best-practice deviations from the article:

- The KAMA filter band is estimated with a **rolling** standard deviation instead of a full-sample statistic.
- Trading signals use **filtered / walk-forward** state probabilities for evaluation, not hindsight-only smoothed states.
- Parameter tuning is left explicit and lightweight rather than hidden behind a black-box optimizer.


## 0. Setup

In [1]:
# Optional: uncomment if this environment is missing dependencies
# %pip install -q numpy pandas matplotlib statsmodels deltalake tables

print('Setup cell ready')


Setup cell ready


## 1. Imports

In [2]:
import warnings
warnings.filterwarnings('ignore')

import os
import math
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.io as pio
import statsmodels.api as sm

os.environ.setdefault('MPLCONFIGDIR', str((pathlib.Path.cwd() / '.mplconfig').resolve()))

plt.rcParams.update({
    'figure.facecolor': '#0f172a',
    'axes.facecolor':   '#0f172a',
    'axes.edgecolor':   '#334155',
    'axes.labelcolor':  'white',
    'xtick.color':      'white',
    'ytick.color':      'white',
    'text.color':       'white',
    'grid.color':       '#334155',
    'grid.linestyle':   '--',
    'grid.alpha':       0.35,
    'legend.facecolor': '#111827',
    'legend.labelcolor':'white',
})

try:
    pio.renderers.default = 'notebook_connected'
except Exception:
    pass

SYMBOL = 'HPG'
ANNUALISE = 252

REGIME_COLORS = {
    'Bullish_Low_Var':  '#22c55e',
    'Bearish_Low_Var':  '#f59e0b',
    'Bullish_High_Var': '#38bdf8',
    'Bearish_High_Var': '#ef4444',
    'Neutral':          '#64748b',
}

print('Imports OK')


Imports OK


## 2. Load Data

In [3]:
REMOTE_HOST = 'http://192.168.1.30:9000'
LOCAL_FILE  = 'stocks_data_latest.h5'

def load_from_delta() -> pd.DataFrame:
    from deltalake import DeltaTable
    storage_options = {
        'AWS_ACCESS_KEY_ID':          'CzOwnLkEDXQy951AOqes',
        'AWS_SECRET_ACCESS_KEY':      'fdRe91TOtqTl0icUkZLsUnWvZa90aZ5qG5rVEf7S',
        'AWS_ENDPOINT_URL':           REMOTE_HOST,
        'AWS_ALLOW_HTTP':             'true',
        'AWS_EC2_METADATA_DISABLED':  'true',
        'AWS_REGION':                 'us-east-1',
        'aws_conditional_put':        'etag',
    }
    start_date = pd.Timestamp.now() - pd.DateOffset(years=10)
    dt = DeltaTable('s3://delta-table-storage/stocks', storage_options=storage_options)
    raw = dt.to_pandas(
        filters=[('date', '>=', start_date), ('symbol', '=', SYMBOL)],
        columns=['symbol', 'date', 'close', 'open', 'high', 'low', 'volume'],
    )
    raw = raw.drop_duplicates(subset=['date', 'symbol'], keep='last')
    raw = raw.set_index(['date', 'symbol']).unstack(level=1)
    return raw


if os.path.exists(LOCAL_FILE):
    print(f'Loading from local HDF5: {LOCAL_FILE}')
    with pd.HDFStore(LOCAL_FILE, mode='r') as store:
        df_all = store['stocks']
else:
    print('HDF5 not found — loading from Delta Lake …')
    df_all = load_from_delta()
    with pd.HDFStore(LOCAL_FILE, mode='w') as store:
        store.put('stocks', df_all)

vn = pd.DataFrame({
    'open':   df_all['open'][SYMBOL],
    'high':   df_all['high'][SYMBOL],
    'low':    df_all['low'][SYMBOL],
    'close':  df_all['close'][SYMBOL],
    'volume': df_all['volume'][SYMBOL],
}).dropna()

vn.index = pd.to_datetime(vn.index)
vn = vn.sort_index()
vn['log_close'] = np.log(vn['close'])
vn['log_return'] = vn['log_close'].diff()
vn['return'] = vn['close'].pct_change()

print(f'Loaded {len(vn):,} bars  |  {vn.index[0].date()} → {vn.index[-1].date()}')
vn.tail(3)


Loading from local HDF5: stocks_data_latest.h5
Loaded 2,498 bars  |  2016-04-19 → 2026-04-17


,open,high,low,close,volume,log_close,log_return,return
date,,,,,,,,
2026-04-15,28.70,28.75,28.05,28.05,29753200.0,3.333989,-0.017668,-0.017513
2026-04-16,28.05,28.20,27.75,27.95,27972100.0,3.330417,-0.003571,-0.003565
2026-04-17,27.95,28.25,27.90,28.00,21399500.0,3.332205,0.001787,0.001789


## 3. Visualise Historical OHLCV

In [4]:
def apply_plotly_theme(fig, title: str, height: int = 800):
    fig.update_layout(
        title=title,
        height=height,
        template='plotly_dark',
        paper_bgcolor='#0f172a',
        plot_bgcolor='#0f172a',
        hovermode='x unified',
        dragmode='pan',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
        margin=dict(l=50, r=30, t=70, b=40),
    )
    fig.update_xaxes(
        showgrid=True,
        gridcolor='#334155',
        rangeslider_visible=False,
        showspikes=True,
        spikecolor='#94a3b8',
        spikesnap='cursor',
    )
    fig.update_yaxes(showgrid=True, gridcolor='#334155')
    return fig


hist_df = vn.tail(180).copy()
candle_colors = np.where(hist_df['close'] >= hist_df['open'], '#22c55e', '#ef4444')

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    row_heights=[0.72, 0.28],
    subplot_titles=(f'{SYMBOL} Candles', 'Volume'),
)

fig.add_trace(
    go.Candlestick(
        x=hist_df.index,
        open=hist_df['open'],
        high=hist_df['high'],
        low=hist_df['low'],
        close=hist_df['close'],
        name='OHLC',
        increasing_line_color='#22c55e',
        decreasing_line_color='#ef4444',
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Bar(
        x=hist_df.index,
        y=hist_df['volume'],
        marker_color=candle_colors,
        opacity=0.75,
        name='Volume',
    ),
    row=2,
    col=1,
)

apply_plotly_theme(fig, title=f'{SYMBOL} — Last 180 bars', height=800)
fig.update_xaxes(rangeslider_visible=True, row=2, col=1)
fig.show()


## 4. Model Configuration

In [5]:
KAMA_WINDOW   = 20
KAMA_FAST     = 2
KAMA_SLOW     = 30
FILTER_GAMMA  = 1.0
FILTER_WINDOW = 20
MARKOV_LAGS   = 1
PROB_THRESHOLD = 0.55
COST_BPS      = 5

TRAIN_BARS = 756   # ~3 years of daily bars
TEST_BARS  = 21    # roughly monthly walk-forward step

print({
    'KAMA_WINDOW': KAMA_WINDOW,
    'KAMA_FAST': KAMA_FAST,
    'KAMA_SLOW': KAMA_SLOW,
    'FILTER_GAMMA': FILTER_GAMMA,
    'FILTER_WINDOW': FILTER_WINDOW,
    'MARKOV_LAGS': MARKOV_LAGS,
    'PROB_THRESHOLD': PROB_THRESHOLD,
    'COST_BPS': COST_BPS,
    'TRAIN_BARS': TRAIN_BARS,
    'TEST_BARS': TEST_BARS,
})


{'KAMA_WINDOW': 20, 'KAMA_FAST': 2, 'KAMA_SLOW': 30, 'FILTER_GAMMA': 1.0, 'FILTER_WINDOW': 20, 'MARKOV_LAGS': 1, 'PROB_THRESHOLD': 0.55, 'COST_BPS': 5, 'TRAIN_BARS': 756, 'TEST_BARS': 21}


## 5. Trend + Volatility Building Blocks

In [6]:
def compute_kama(series: pd.Series, window: int = 10, fast: int = 2, slow: int = 30) -> pd.Series:
    series = pd.Series(series, dtype=float)
    change = series.diff(window).abs()
    volatility = series.diff().abs().rolling(window).sum()
    efficiency_ratio = (change / volatility.replace(0, np.nan)).clip(lower=0, upper=1)

    fast_sc = 2.0 / (fast + 1.0)
    slow_sc = 2.0 / (slow + 1.0)
    smoothing_constant = (efficiency_ratio * (fast_sc - slow_sc) + slow_sc) ** 2

    kama = pd.Series(np.nan, index=series.index, dtype=float)
    if len(series) <= window:
        return kama

    kama.iloc[window] = series.iloc[: window + 1].mean()
    for i in range(window + 1, len(series)):
        alpha = smoothing_constant.iloc[i]
        prev = kama.iloc[i - 1]
        kama.iloc[i] = prev if np.isnan(alpha) else prev + alpha * (series.iloc[i] - prev)
    return kama


def build_trend_overlay(
    log_close: pd.Series,
    kama_window: int = 10,
    kama_fast: int = 2,
    kama_slow: int = 30,
    gamma: float = 1.0,
    filter_window: int | None = None,
) -> pd.DataFrame:
    filter_window = kama_window if filter_window is None else filter_window

    kama = compute_kama(log_close, window=kama_window, fast=kama_fast, slow=kama_slow)
    period_low = kama.rolling(kama_window, min_periods=kama_window).min().shift(1)
    period_high = kama.rolling(kama_window, min_periods=kama_window).max().shift(1)

    # Rolling band avoids the full-sample leakage implied by a single global std estimate.
    filter_band = gamma * kama.diff(kama_window).rolling(filter_window, min_periods=filter_window).std()

    trend = pd.Series(0, index=log_close.index, dtype='int64')
    trend[kama > period_low + filter_band] = 1
    trend[kama < period_high - filter_band] = -1

    return pd.DataFrame({
        'kama': kama,
        'period_low': period_low,
        'period_high': period_high,
        'filter_band': filter_band,
        'trend': trend,
    })


def prepare_markov_inputs(log_returns: pd.Series, lag_order: int = 1):
    y = pd.Series(log_returns, dtype=float).dropna().rename('ret')
    exog = pd.concat({f'lag_{lag}': y.shift(lag) for lag in range(1, lag_order + 1)}, axis=1)
    data = pd.concat([y, exog], axis=1).dropna()
    return data['ret'], data.drop(columns='ret')


def label_states_by_variance(returns: pd.Series, probs: pd.DataFrame) -> dict:
    aligned_returns = returns.loc[probs.index]
    state_var = {}
    for state in probs.columns:
        weights = probs[state].clip(lower=1e-12)
        mean_ret = np.average(aligned_returns, weights=weights)
        state_var[state] = np.average((aligned_returns - mean_ret) ** 2, weights=weights)
    ordered = sorted(state_var, key=state_var.get)
    return {'low': ordered[0], 'high': ordered[-1], 'state_var': state_var}


def fit_markov_volatility(log_returns: pd.Series, lag_order: int = 1):
    endog, exog = prepare_markov_inputs(log_returns, lag_order=lag_order)
    model = sm.tsa.MarkovRegression(
        endog,
        k_regimes=2,
        trend='n',
        exog=exog,
        switching_variance=True,
    )
    result = model.fit(disp=False)
    mapping = label_states_by_variance(endog, result.smoothed_marginal_probabilities)

    smoothed_probs = pd.DataFrame({
        'low_var_prob': result.smoothed_marginal_probabilities[mapping['low']],
        'high_var_prob': result.smoothed_marginal_probabilities[mapping['high']],
    })
    filtered_probs = pd.DataFrame({
        'low_var_prob': result.filtered_marginal_probabilities[mapping['low']],
        'high_var_prob': result.filtered_marginal_probabilities[mapping['high']],
    })

    return {
        'result': result,
        'mapping': mapping,
        'smoothed_probs': smoothed_probs,
        'filtered_probs': filtered_probs,
    }


def filter_markov_probs(log_returns: pd.Series, params, mapping: dict, lag_order: int = 1) -> pd.DataFrame:
    endog, exog = prepare_markov_inputs(log_returns, lag_order=lag_order)
    model = sm.tsa.MarkovRegression(
        endog,
        k_regimes=2,
        trend='n',
        exog=exog,
        switching_variance=True,
    )
    result = model.filter(params)
    return pd.DataFrame({
        'low_var_prob': result.filtered_marginal_probabilities[mapping['low']],
        'high_var_prob': result.filtered_marginal_probabilities[mapping['high']],
    })


trend_df = build_trend_overlay(
    vn['log_close'],
    kama_window=KAMA_WINDOW,
    kama_fast=KAMA_FAST,
    kama_slow=KAMA_SLOW,
    gamma=FILTER_GAMMA,
    filter_window=FILTER_WINDOW,
)
markov_fit = fit_markov_volatility(vn['log_return'], lag_order=MARKOV_LAGS)

print('Trend + volatility features ready')
print('State variance ranking:', markov_fit['mapping']['state_var'])
trend_df.tail(3)


/Users/phuchuynh/.pyenv/versions/3.12.12/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning:

A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.



Trend + volatility features ready
State variance ranking: {0: np.float64(0.00017025841880317883), 1: np.float64(0.0009677403944962289)}


,kama,period_low,period_high,filter_band,trend
date,,,,,
2026-04-15,3.302206,3.295885,3.307643,0.009246,0
2026-04-16,3.302889,3.295885,3.307507,0.008101,0
2026-04-17,3.304522,3.295885,3.306741,0.006832,1


## 6. Combine Into Four Regimes

In [7]:
def combine_regimes(index_like, vol_probs: pd.DataFrame, trend_data: pd.DataFrame, prob_threshold: float = 0.55) -> pd.DataFrame:
    out = pd.DataFrame(index=index_like)
    out = out.join(vol_probs[['low_var_prob', 'high_var_prob']], how='left')
    out = out.join(trend_data[['kama', 'trend']], how='left')

    out['label'] = 'Neutral'
    low_var = out['low_var_prob'] >= prob_threshold
    high_var = out['high_var_prob'] >= prob_threshold
    bullish = out['trend'] > 0
    bearish = out['trend'] < 0

    out.loc[low_var & bullish, 'label'] = 'Bullish_Low_Var'
    out.loc[low_var & bearish, 'label'] = 'Bearish_Low_Var'
    out.loc[high_var & bullish, 'label'] = 'Bullish_High_Var'
    out.loc[high_var & bearish, 'label'] = 'Bearish_High_Var'

    out['color'] = out['label'].map(REGIME_COLORS)
    out['state_id'] = (out['label'] != out['label'].shift()).cumsum()
    return out


smoothed_regimes = combine_regimes(vn.index, markov_fit['smoothed_probs'], trend_df, prob_threshold=PROB_THRESHOLD)
filtered_regimes = combine_regimes(vn.index, markov_fit['filtered_probs'], trend_df, prob_threshold=PROB_THRESHOLD)

summary = pd.concat([
    smoothed_regimes['label'].value_counts().rename('smoothed_count'),
    filtered_regimes['label'].value_counts().rename('filtered_count'),
], axis=1).fillna(0).astype(int)

summary


,smoothed_count,filtered_count
label,,
Bullish_Low_Var,869,880
Neutral,676,714
Bearish_Low_Var,413,419
Bullish_High_Var,294,262
Bearish_High_Var,246,223


## 7. Visualise Regimes

In [8]:
PLOT_BARS = 500
plot_df = vn.tail(PLOT_BARS).copy()
plot_smoothed = smoothed_regimes.loc[plot_df.index]
plot_filtered = filtered_regimes.loc[plot_df.index]
plot_probs = markov_fit['filtered_probs'].reindex(plot_df.index)
plot_trend = trend_df.reindex(plot_df.index)

fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.04,
    row_heights=[0.58, 0.24, 0.18],
    subplot_titles=(
        f'{SYMBOL}',
    ),
)

fig.add_trace(
    go.Scatter(
        x=plot_df.index,
        y=plot_df['close'],
        mode='lines',
        line=dict(color='white', width=1.4),
        name='Close',
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=plot_df.index,
        y=np.exp(plot_trend['kama']),
        mode='lines',
        line=dict(color='#38bdf8', width=1.2),
        name='KAMA (exp back to price)',
    ),
    row=1,
    col=1,
)

for label, color in REGIME_COLORS.items():
    mask = plot_filtered['label'] == label
    if mask.any():
        fig.add_trace(
            go.Scatter(
                x=plot_df.index[mask],
                y=plot_df.loc[mask, 'close'],
                mode='markers',
                marker=dict(size=7, color=color),
                name=label,
            ),
            row=1,
            col=1,
        )

fig.add_trace(
    go.Scatter(
        x=plot_probs.index,
        y=plot_probs['low_var_prob'],
        mode='lines',
        line=dict(color='#22c55e', width=1.2),
        name='Low variance prob',
    ),
    row=2,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=plot_probs.index,
        y=plot_probs['high_var_prob'],
        mode='lines',
        line=dict(color='#ef4444', width=1.2),
        name='High variance prob',
    ),
    row=2,
    col=1,
)

fig.add_hline(
    y=PROB_THRESHOLD,
    line_dash='dash',
    line_color='#f8fafc',
    opacity=0.6,
    annotation_text='Decision threshold',
    row=2,
    col=1,
)

trend_color = plot_trend['trend'].map({1: '#22c55e', -1: '#ef4444', 0: '#64748b'})
fig.add_trace(
    go.Bar(
        x=plot_trend.index,
        y=plot_trend['trend'],
        marker_color=trend_color,
        name='Trend state',
    ),
    row=3,
    col=1,
)

apply_plotly_theme(fig, title='', height=950)
fig.update_yaxes(title_text='Index Level', row=1, col=1)
fig.update_yaxes(title_text='Probability', range=[-0.02, 1.02], row=2, col=1)
fig.update_yaxes(title_text='Trend', tickvals=[-1, 0, 1], ticktext=['Bear', 'Flat', 'Bull'], row=3, col=1)
fig.update_xaxes(rangeslider_visible=True, row=3, col=1)
fig.show()


## 8. Strategy Construction

In [9]:
def build_positions(labels: pd.Series, mode: str = 'long_short') -> pd.Series:
    """Long is sticky: enter on Bullish_Low_Var, stay long until Bearish_Low_Var or Bearish_High_Var.

    In long_short mode, Bearish_High_Var is short (-1) after long is cleared; Bearish_Low_Var is flat.
    """
    if mode not in ('long_short', 'long_only'):
        raise ValueError(f'Unsupported mode: {mode}')
    positions = pd.Series(0.0, index=labels.index)
    in_long = False
    for i, lab in enumerate(labels):
        if lab == 'Bullish_Low_Var':
            in_long = True
        elif lab in ('Bearish_Low_Var', 'Bearish_High_Var'):
            in_long = False
        if in_long:
            positions.iloc[i] = 1.0
        elif lab == 'Bearish_High_Var' and mode == 'long_short':
            positions.iloc[i] = -1.0
        elif lab == 'Bearish_High_Var' and mode == 'long_only':
            positions.iloc[i] = 0.0
    return positions


def max_drawdown(equity_curve: pd.Series) -> float:
    running_max = equity_curve.cummax()
    drawdown = equity_curve / running_max - 1.0
    return float(drawdown.min())


def performance_stats(returns: pd.Series, periods_per_year: int = 252) -> pd.Series:
    returns = pd.Series(returns, dtype=float).dropna()
    if returns.empty:
        return pd.Series(dtype=float)

    equity = (1.0 + returns).cumprod()
    total_return = equity.iloc[-1] - 1.0
    cagr = equity.iloc[-1] ** (periods_per_year / len(returns)) - 1.0
    vol = returns.std() * np.sqrt(periods_per_year)
    sharpe = np.nan if returns.std() == 0 else returns.mean() / returns.std() * np.sqrt(periods_per_year)

    downside = returns[returns < 0].std() * np.sqrt(periods_per_year)
    sortino = np.nan if downside == 0 or np.isnan(downside) else returns.mean() * periods_per_year / downside

    mdd = max_drawdown(equity)
    calmar = np.nan if mdd == 0 else cagr / abs(mdd)

    return pd.Series({
        'Total Return': total_return,
        'CAGR': cagr,
        'Volatility': vol,
        'Sharpe': sharpe,
        'Sortino': sortino,
        'Max Drawdown': mdd,
        'Calmar': calmar,
    })


def backtest_from_regimes(close: pd.Series, regime_table: pd.DataFrame, mode: str = 'long_short', cost_bps: float = 5.0):
    positions = build_positions(regime_table['label'], mode=mode)
    returns = close.pct_change().fillna(0.0)

    turnover = positions.diff().abs().fillna(positions.abs())
    costs = turnover * (cost_bps / 10_000.0)
    strategy_returns = positions.shift(1).fillna(0.0) * returns - costs

    result = pd.DataFrame({
        'close_return': returns,
        'position': positions,
        'turnover': turnover,
        'cost': costs,
        'strategy_return': strategy_returns,
        'equity_curve': (1.0 + strategy_returns).cumprod(),
        'buy_hold_curve': (1.0 + returns).cumprod(),
    }, index=close.index)
    return result


static_ls = backtest_from_regimes(vn['close'], filtered_regimes, mode='long_short', cost_bps=COST_BPS)
static_lo = backtest_from_regimes(vn['close'], filtered_regimes, mode='long_only', cost_bps=COST_BPS)

pd.concat({
    'Buy & Hold': performance_stats(static_ls['close_return']),
    'Regime Long/Short': performance_stats(static_ls['strategy_return']),
    'Regime Long-Only': performance_stats(static_lo['strategy_return']),
}, axis=1).T.style.format('{:.2%}', subset=['Total Return', 'CAGR', 'Volatility', 'Max Drawdown']).format('{:.2f}', subset=['Sharpe', 'Sortino', 'Calmar'])


,Total Return,CAGR,Volatility,Sharpe,Sortino,Max Drawdown,Calmar
Buy & Hold,809.09%,24.94%,33.04%,0.84,1.23,-72.42%,0.34
Regime Long/Short,44.68%,3.80%,22.73%,0.28,0.26,-54.66%,0.07
Regime Long-Only,144.94%,9.46%,16.50%,0.63,0.56,-30.80%,0.31


## 9. Strategy Curves

In [10]:
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    row_heights=[0.72, 0.28],
    subplot_titles=(
        f'{SYMBOL} — In-Sample Strategy Curves (Filtered Probabilities)',
        'Positioning',
    ),
)

fig.add_trace(
    go.Scatter(
        x=static_ls.index,
        y=static_ls['buy_hold_curve'],
        mode='lines',
        line=dict(color='#94a3b8', width=1.3),
        name='Buy & Hold',
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=static_ls.index,
        y=static_ls['equity_curve'],
        mode='lines',
        line=dict(color='#22c55e', width=1.4),
        name='Regime Long/Short',
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=static_lo.index,
        y=static_lo['equity_curve'],
        mode='lines',
        line=dict(color='#38bdf8', width=1.2),
        name='Regime Long-Only',
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=static_ls.index,
        y=static_ls['position'],
        mode='lines',
        line=dict(color='#f59e0b', width=1.0),
        name='Long/Short position',
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=static_lo.index,
        y=static_lo['position'],
        mode='lines',
        line=dict(color='#38bdf8', width=1.0),
        name='Long-only position',
    ),
    row=2,
    col=1,
)

apply_plotly_theme(fig, title=f'{SYMBOL} — In-Sample Strategy Curves', height=850)
fig.update_yaxes(title_text='Equity (growth of 1)', row=1, col=1)
fig.update_yaxes(title_text='Position', tickvals=[-1, 0, 1], row=2, col=1)
fig.update_xaxes(rangeslider_visible=True, row=2, col=1)
fig.show()


## 10. Walk-Forward Evaluation

This is the important part for tradability.

For each out-of-sample block:

1. Fit the Markov model on the trailing `TRAIN_BARS` window.
2. Freeze the fitted parameters.
3. Apply the Hamilton filter to `train + test` data with those fixed parameters.
4. Keep only the **test-period filtered probabilities**.
5. Combine them with the KAMA trend overlay and trade the resulting regime labels.


In [11]:
def walk_forward_regime_backtest(
    close: pd.Series,
    log_close: pd.Series,
    kama_window: int,
    kama_fast: int,
    kama_slow: int,
    gamma: float,
    filter_window: int,
    markov_lags: int,
    prob_threshold: float,
    train_bars: int,
    test_bars: int,
    cost_bps: float,
    mode: str = 'long_short',
):
    log_returns = log_close.diff().dropna()
    trend_overlay = build_trend_overlay(
        log_close,
        kama_window=kama_window,
        kama_fast=kama_fast,
        kama_slow=kama_slow,
        gamma=gamma,
        filter_window=filter_window,
    )

    wf_probs = pd.DataFrame(index=close.index, columns=['low_var_prob', 'high_var_prob'], dtype=float)
    blocks = []

    for start in range(train_bars, len(log_returns), test_bars):
        train_slice = log_returns.iloc[start - train_bars:start]
        test_slice = log_returns.iloc[start:start + test_bars]
        if len(test_slice) < max(5, markov_lags + 2):
            break

        try:
            train_fit = fit_markov_volatility(train_slice, lag_order=markov_lags)
            combined_slice = pd.concat([train_slice, test_slice])
            filtered_probs = filter_markov_probs(
                combined_slice,
                params=train_fit['result'].params,
                mapping=train_fit['mapping'],
                lag_order=markov_lags,
            )
            filtered_probs = filtered_probs.loc[test_slice.index]
            wf_probs.loc[filtered_probs.index, ['low_var_prob', 'high_var_prob']] = filtered_probs[['low_var_prob', 'high_var_prob']].values
            blocks.append({
                'train_start': train_slice.index[0],
                'train_end': train_slice.index[-1],
                'test_start': test_slice.index[0],
                'test_end': test_slice.index[-1],
                'bars': len(test_slice),
            })
        except Exception as exc:
            print(f'Walk-forward block failed at {test_slice.index[0].date()}: {exc}')

    wf_regimes = combine_regimes(close.index, wf_probs, trend_overlay, prob_threshold=prob_threshold)
    wf_bt = backtest_from_regimes(close, wf_regimes, mode=mode, cost_bps=cost_bps)
    return wf_regimes, wf_bt, pd.DataFrame(blocks)


wf_regimes_ls, wf_bt_ls, wf_blocks = walk_forward_regime_backtest(
    close=vn['close'],
    log_close=vn['log_close'],
    kama_window=KAMA_WINDOW,
    kama_fast=KAMA_FAST,
    kama_slow=KAMA_SLOW,
    gamma=FILTER_GAMMA,
    filter_window=FILTER_WINDOW,
    markov_lags=MARKOV_LAGS,
    prob_threshold=PROB_THRESHOLD,
    train_bars=TRAIN_BARS,
    test_bars=TEST_BARS,
    cost_bps=COST_BPS,
    mode='long_short',
)

wf_regimes_lo, wf_bt_lo, _ = walk_forward_regime_backtest(
    close=vn['close'],
    log_close=vn['log_close'],
    kama_window=KAMA_WINDOW,
    kama_fast=KAMA_FAST,
    kama_slow=KAMA_SLOW,
    gamma=FILTER_GAMMA,
    filter_window=FILTER_WINDOW,
    markov_lags=MARKOV_LAGS,
    prob_threshold=PROB_THRESHOLD,
    train_bars=TRAIN_BARS,
    test_bars=TEST_BARS,
    cost_bps=COST_BPS,
    mode='long_only',
)

print(f'Walk-forward blocks: {len(wf_blocks)}')
wf_blocks.tail()


/Users/phuchuynh/.pyenv/versions/3.12.12/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning:

A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.

/Users/phuchuynh/.pyenv/versions/3.12.12/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning:

A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.

/Users/phuchuynh/.pyenv/versions/3.12.12/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning:

A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.

/Users/phuchuynh/.pyenv/versions/3.12.12/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning:

A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.



Walk-forward blocks: 83


/Users/phuchuynh/.pyenv/versions/3.12.12/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning:

A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.



,train_start,train_end,test_start,test_end,bars
78,2022-11-04,2025-11-14,2025-11-17,2025-12-15,21
79,2022-12-05,2025-12-15,2025-12-16,2026-01-15,21
80,2023-01-04,2026-01-15,2026-01-16,2026-02-13,21
81,2023-02-09,2026-02-13,2026-02-23,2026-03-23,21
82,2023-03-10,2026-03-23,2026-03-24,2026-04-17,19


In [12]:
wf_summary = pd.concat({
    'Buy & Hold': performance_stats(wf_bt_ls['close_return']),
    'WF Regime Long/Short': performance_stats(wf_bt_ls['strategy_return']),
    'WF Regime Long-Only': performance_stats(wf_bt_lo['strategy_return']),
}, axis=1).T

wf_summary.style.format('{:.2%}', subset=['Total Return', 'CAGR', 'Volatility', 'Max Drawdown']).format('{:.2f}', subset=['Sharpe', 'Sortino', 'Calmar'])


,Total Return,CAGR,Volatility,Sharpe,Sortino,Max Drawdown,Calmar
Buy & Hold,809.09%,24.94%,33.04%,0.84,1.23,-72.42%,0.34
WF Regime Long/Short,36.92%,3.22%,19.71%,0.26,0.20,-44.62%,0.07
WF Regime Long-Only,76.61%,5.91%,13.43%,0.49,0.36,-31.63%,0.19


In [13]:
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    row_heights=[0.72, 0.28],
    subplot_titles=(
        f'{SYMBOL} — Walk-Forward Equity Curves',
        'Recent Walk-Forward Positioning',
    ),
)

fig.add_trace(
    go.Scatter(
        x=wf_bt_ls.index,
        y=wf_bt_ls['buy_hold_curve'],
        mode='lines',
        line=dict(color='#94a3b8', width=1.3),
        name='Buy & Hold',
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=wf_bt_ls.index,
        y=wf_bt_ls['equity_curve'],
        mode='lines',
        line=dict(color='#22c55e', width=1.4),
        name='WF Long/Short',
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=wf_bt_lo.index,
        y=wf_bt_lo['equity_curve'],
        mode='lines',
        line=dict(color='#38bdf8', width=1.2),
        name='WF Long-Only',
    ),
    row=1,
    col=1,
)

wf_tail = 350
plot_idx = wf_bt_ls.tail(wf_tail).index

fig.add_trace(
    go.Scatter(
        x=plot_idx,
        y=wf_bt_ls.loc[plot_idx, 'position'],
        mode='lines',
        line=dict(color='#f59e0b', width=1.0),
        name='WF Long/Short position',
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=plot_idx,
        y=wf_bt_lo.loc[plot_idx, 'position'],
        mode='lines',
        line=dict(color='#38bdf8', width=1.0),
        name='WF Long-only position',
    ),
    row=2,
    col=1,
)

apply_plotly_theme(fig, title=f'{SYMBOL} — Walk-Forward Strategy Curves', height=850)
fig.update_yaxes(title_text='Equity (growth of 1)', row=1, col=1)
fig.update_yaxes(title_text='Position', tickvals=[-1, 0, 1], row=2, col=1)
fig.update_xaxes(rangeslider_visible=True, row=2, col=1)
fig.show()


## 11. Readout

Interpretation guide:

- `Bullish_Low_Var` is the regime the article treats as the cleanest long setup.
- `Bearish_High_Var` is the risk-off / short regime.
- `Bullish_High_Var` and `Bearish_Low_Var` are informative but often less attractive for simple directional trading.
- The walk-forward section is the more realistic benchmark; the earlier in-sample charts are mainly for intuition and diagnostics.

Natural next steps if you want to extend this notebook:

- Replace fixed parameters with a small walk-forward grid search.
- Test on tradable VN30 / ETF proxies instead of the index itself.
- Add volatility targeting or position scaling by state probability.
- Compare Markov-only, KAMA-only, and combined KAMA+MSR rules side by side.
